# SDS PySpark Tutorial — Part 1
## Foundations for a Complete Beginner

**Goal:** build the mental model you need *before* learning lots of Spark syntax.

This notebook assumes:

- you know basic Python;
- you may have used pandas/Jupyter;
- you have **not** yet developed an intuitive model of distributed computing;
- you are studying Spark specifically for the **Scalable Data Science comprehensive exam**.

This is **not** a general Spark certification course. We will learn only the concepts needed to understand and modify the SDS exam notebooks.

---

## By the end of Part 1, you should be able to explain in plain English:

1. Why Spark exists.
2. What the **driver**, **executors**, and **partitions** are.
3. What “distributed” means.
4. Why a Spark DataFrame can be much larger than the memory of your Python process.
5. What transformations and actions are.
6. Why Spark uses lazy evaluation.
7. Exactly why `collect()` can be dangerous.
8. How `show()`, `take()`, `first()`, `count()`, and `collect()` differ.
9. How to create, inspect, select, filter, and transform DataFrames.
10. How `groupBy` changes the shape of data.

**Important:** the first checkpoint comes only after these ideas have been explained slowly.

# 1. Why does Spark exist?

Start with the simplest possible data-processing model.

Suppose your laptop has **16 GB of RAM** and you have a **500 MB** CSV file.

Ordinary Python or pandas may be perfectly reasonable:

```text
500 MB file
    ↓
load into one computer
    ↓
process in Python/pandas
    ↓
result
```

Now imagine the dataset is **500 GB**.

Your machine still has 16 GB RAM.

The obvious approach breaks:

```text
500 GB dataset
    ↓
try to load into 16 GB machine
    ↓
does not fit
```

This is the problem distributed systems such as Spark are designed to help solve.

## 1.1 The core idea of distributed processing

Instead of demanding that **one computer** hold all the data and do all the work, divide the data into pieces and process those pieces in parallel.

Conceptually:

```text
                   500 GB DATASET
                        |
          +-------------+-------------+
          |             |             |
        chunk 1       chunk 2       chunk 3
          |             |             |
       worker 1      worker 2      worker 3
```

Each worker handles part of the data.

The result may then be combined.

This gives us the first sentence worth remembering:

> **Spark exists so that large data and large computations can remain distributed rather than being forced onto one Python process.**

That sentence will later explain why `collect()` can be dangerous.

## 1.2 Spark does not make every algorithm magically cheap

Distribution solves an important problem, but not every problem.

Suppose a graph has one million nodes and you decide to compare **every node with every other node**.

That is roughly:

\[
10^6 \times 10^6 = 10^{12}
\]

ordered comparisons.

Spark can distribute work, but \(10^{12}\) comparisons are still enormous.

So there are **two different questions** in the SDS exam:

1. **Correctness:** does the algorithm compute the right thing?
2. **Scalability:** is the expensive state/computation kept distributed and is the algorithm itself reasonable at scale?

A correct local algorithm can still be a weak Scalable Data Science answer.

# 2. The three objects you need first: driver, executors, partitions

We will introduce them one at a time.

Do not memorize the words yet. Build a picture.

## 2.1 The driver: the coordinator

The **driver** is the process running your Spark application.

In a Jupyter workflow, you can think of your notebook/Python program as controlling the driver.

A useful analogy:

```text
DRIVER = project manager
```

The project manager:

- receives your instructions;
- creates a plan;
- assigns work;
- tracks the work;
- receives small final results.

The project manager should **not** personally carry every box in the warehouse.

Likewise, the Spark driver should generally not hold the entire huge dataset.

## 2.2 Executors: the workers

Executors are processes that perform distributed tasks.

Analogy:

```text
DRIVER    = project manager
EXECUTOR  = worker
```

If you ask Spark to filter a huge DataFrame, the conceptual idea is not:

```text
driver checks row 1
driver checks row 2
driver checks row 3
...
```

Instead, different workers can process different pieces of the data.

## 2.3 Partitions: the pieces of data/work

A Spark DataFrame may look like one table:

| id | amount |
|---|---:|
| A | 10 |
| B | 20 |
| C | 30 |
| ... | ... |

But Spark can store/process that logical table as multiple **partitions**.

For example, one million rows might conceptually be divided as:

```text
Partition 1 → rows 1 ... 250,000
Partition 2 → rows 250,001 ... 500,000
Partition 3 → rows 500,001 ... 750,000
Partition 4 → rows 750,001 ... 1,000,000
```

A partition is a unit that Spark can schedule as work.

The exact physical placement is more complicated in a real cluster, but this model is enough for the exam.

## 2.4 Put the three concepts together

```text
                         DRIVER
                  "Filter amount > 100"
                           |
              creates / schedules the plan
                           |
        +------------------+------------------+
        |                  |                  |
     EXECUTOR           EXECUTOR           EXECUTOR
        |                  |                  |
   partition(s)       partition(s)       partition(s)
        |                  |                  |
   filter locally     filter locally     filter locally
```

The important point is:

> **The large DataFrame can stay distributed while the driver coordinates the computation.**

This is the mental model behind most of the Spark code in the SDS notebooks.

# 3. What is a Spark DataFrame really?

A Spark DataFrame is a **logical distributed table** with named columns and a schema.

It is similar in appearance to a pandas DataFrame, but the execution model is very different.

With pandas, you typically have something like:

```text
Python process
     |
entire pandas DataFrame
```

With Spark, think:

```text
Python/driver
     |
logical DataFrame definition
     |
distributed partitions on workers
```

This difference is why code that *looks* similar can behave very differently.

## 3.1 A DataFrame variable does not necessarily contain all rows in Python

Suppose you write:

```python
df = spark.read.csv("huge.csv")
```

The Python variable `df` is **not** normally a Python list containing every row.

It represents a Spark DataFrame and its distributed computation.

That is why a Spark DataFrame can represent data much larger than your notebook process could hold by itself.

# 4. Transformations: describing what you want Spark to do

Suppose you write:

```python
filtered = df.filter(F.col("amount") > 100)
```

You have described a new DataFrame:

> “Take `df` and keep only rows where amount > 100.”

`filter()` is a **transformation**.

Other common transformations include:

- `select`
- `withColumn`
- `join`
- `groupBy(...).agg(...)`
- `distinct`

A transformation creates a description of a new distributed result.

## 4.1 Transformations are usually lazy

Spark generally does **not** immediately execute the full pipeline every time you type a transformation.

Suppose you write:

```python
x = df.filter(F.col("amount") > 100)
y = x.select("user_id", "amount")
z = y.groupBy("user_id").sum("amount")
```

Think of Spark building a plan:

```text
READ DATA
   ↓
FILTER amount > 100
   ↓
SELECT user_id, amount
   ↓
GROUP BY user_id
   ↓
SUM amount
```

At this stage Spark knows **what** you want.

# 5. Actions: asking Spark to produce a result now

An **action** forces Spark to execute enough of the plan to return or display a result.

Examples:

```python
z.show()
z.count()
z.collect()
z.first()
z.take(5)
```

So the most useful beginner distinction is:

\[
\boxed{\text{Transformation = describe work}}
\]

\[
\boxed{\text{Action = ask Spark to perform work}}
\]

This is much more useful than memorizing two lists with no intuition.

## 5.1 Why lazy evaluation is useful

Because Spark sees a larger portion of the pipeline before executing it, it can plan the work.

Conceptually, instead of blindly doing:

```text
operation 1 → materialize everything
operation 2 → materialize everything again
operation 3 → materialize everything again
```

Spark can reason about a combined plan.

For the exam, you do **not** need to become an expert in the Catalyst optimizer.

You only need to understand:

> A chain of transformations describes a plan; an action triggers execution.

# 6. Now we can explain `collect()` properly

This is the idea that was introduced too early in the previous notebook.

Suppose the distributed dataset is:

```text
Executor 1 has about 2 GB
Executor 2 has about 2 GB
Executor 3 has about 2 GB
Executor 4 has about 2 GB
```

The total logical DataFrame is about **8 GB**.

The work is distributed.

Now you execute:

```python
rows = df.collect()
```

What are you asking Spark to do?

Conceptually:

```text
Executor 1 ──┐
Executor 2 ──┤
Executor 3 ──┼────→ DRIVER / PYTHON
Executor 4 ──┘
```

You are asking:

> **“Send all rows of this DataFrame back to my driver process.”**

## 6.1 Why can that be dangerous?

Imagine:

- distributed DataFrame = 8 GB;
- memory comfortably available to your Python/driver process = 2 GB.

The DataFrame may be perfectly manageable **while distributed**.

But `collect()` asks the driver to receive the entire result.

That can cause:

- driver memory exhaustion;
- very slow network transfer;
- JVM/Python memory pressure;
- crashes;
- loss of scalability.

The key sentence is:

\[
\boxed{\text{Spark distributes data; collect() brings the result back to the driver.}}
\]

`collect()` is not “bad.”

It is bad **when the result is not known to be small**.

## 6.2 Safe versus unsafe examples

Usually reasonable:

```python
top10 = scores.orderBy(F.desc("score")).limit(10).collect()
```

You deliberately limited the result to 10 rows.

Also reasonable:

```python
n = df.count()
```

The distributed computation may be huge, but the result returned to the driver is just one number.

Potentially dangerous:

```python
all_edges = edges.collect()
```

if `edges` contains hundreds of thousands or millions of rows.

Then doing this:

```python
adj = {}
for e in all_edges:
    ...
```

means the expensive graph state is now local Python state.

# 7. `show`, `take`, `first`, `count`, and `collect`

These are all actions, but they return very different amounts of information.

| Command | What comes back / is displayed | Typical beginner use |
|---|---|---|
| `df.show(10)` | displays a small number of rows | inspect data |
| `df.first()` | one `Row` | inspect one result/scalar row |
| `df.take(10)` | at most 10 `Row` objects | use a tiny result in Python |
| `df.count()` | one integer | count rows |
| `df.collect()` | **all rows** | only when result is intentionally small |

The most important point:

> “Action” does **not** automatically mean “dangerous.”  
> The question is also **how much data returns to the driver**.

## Guided Checkpoint 1 — now the questions are answerable

### Q1
You have a 50-million-row DataFrame and only want to inspect ten rows.

Which is safer?

A. `df.collect()`  
B. `df.show(10)`

**Hint:** Which command asks for every row?

<details>
<summary>Answer</summary>

**B. `df.show(10)`**. `collect()` requests all rows.
</details>

### Q2
You write:

```python
x = df.filter(F.col("score") > 10)
```

What kind of operation is `filter()`?

A. transformation  
B. action

<details>
<summary>Answer</summary>

**A. transformation.** It describes a new DataFrame.
</details>

### Q3
Why can `collect()` be dangerous?

Choose the best explanation:

A. It always deletes partitions.  
B. It sends the complete result to the driver, which may not have enough memory.  
C. It changes integers into strings.

<details>
<summary>Answer</summary>

**B.**
</details>

## Independent Checkpoint 1B

Now answer without choices:

1. In one sentence, what is the driver?
2. What is an executor?
3. What is a partition?
4. What is the difference between a transformation and an action?
5. Why can an 8 GB DataFrame be manageable in Spark but dangerous after `collect()`?
6. When is `collect()` reasonable?

If you cannot explain these comfortably, reread Sections 1–7 before continuing.

# 8. Start Spark

Everything after this point becomes hands-on.

If your exam/server already created `spark`, the following call simply obtains/creates the session.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("SDS-PySpark-Part1").getOrCreate()

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

# 9. Create a tiny DataFrame

We will deliberately start with tiny data so you can predict every result.

Imagine these are security events.

In [ ]:
events_data = [
    ("u1", "login",    "PH", 10.0),
    ("u1", "download", "PH", 50.0),
    ("u2", "login",    "SG", 12.0),
    ("u2", "download", "SG", 70.0),
    ("u3", "login",    "PH",  9.0),
    ("u3", "download", None, 80.0),
]

schema = T.StructType([
    T.StructField("user_id", T.StringType(), False),
    T.StructField("action", T.StringType(), False),
    T.StructField("country", T.StringType(), True),
    T.StructField("bytes_mb", T.DoubleType(), False),
])

events = spark.createDataFrame(events_data, schema)
events.show(truncate=False)

## 9.1 Read the schema

Before running:

```python
events.printSchema()
```

predict:

- which columns are strings?
- which one is numeric?
- which column allows null?

In [ ]:
events.printSchema()

### Why schema matters in an exam

Immediately after loading unfamiliar data, inspect the schema.

Many bugs are actually data-type bugs:

- numeric-looking IDs inferred as numbers;
- timestamps left as strings;
- count fields read as strings;
- null values ignored.

For exam work, `inferSchema=True` may be convenient for a small CSV, but you should still inspect the result.

# 10. `select`: choose columns

Start with the simplest DataFrame transformation.

In [ ]:
selected = events.select("user_id", "bytes_mb")

# selected is a new DataFrame description.
selected.show()

Notice the sequence:

```python
selected = events.select(...)
```

Transformation: describe a two-column DataFrame.

```python
selected.show()
```

Action: execute enough of the plan to display rows.

## 10.1 Column expressions

Spark lets you build expressions over columns.

Predict the new column before running:

In [ ]:
events.select(
    "user_id",
    "bytes_mb",
    (F.col("bytes_mb") * 1024).alias("bytes_kb")
).show()

# 11. `withColumn`: add or replace a column

Spark DataFrames are conceptually immutable.

This:

```python
events2 = events.withColumn(...)
```

creates a new DataFrame definition.

It does not edit Python rows one at a time.

In [ ]:
events2 = events.withColumn(
    "is_large",
    F.col("bytes_mb") >= 50
)

events2.show()

# 12. `filter`: keep rows matching a condition

Predict first: which rows have `bytes_mb >= 50`?

In [ ]:
large = events.filter(F.col("bytes_mb") >= 50)
large.show()

## 12.1 Multiple conditions

For Spark Column expressions use:

- `&` for AND
- `|` for OR
- `~` for NOT

Use parentheses around each condition.

In [ ]:
events.filter(
    (F.col("country") == "PH") &
    (F.col("bytes_mb") >= 10)
).show()

### Common beginner mistake

Wrong:

```python
df.filter((F.col("x") > 0) and (F.col("y") > 0))
```

Right:

```python
df.filter((F.col("x") > 0) & (F.col("y") > 0))
```

# 13. Null values

One row has `country = None`.

In Spark/SQL logic, null means “missing/unknown,” not an ordinary string.

Useful operations:

- `.isNull()`
- `.isNotNull()`
- `.fillna(...)`
- `F.coalesce(...)`

In [ ]:
events.filter(F.col("country").isNull()).show()

In [ ]:
events.fillna({"country": "UNKNOWN"}).show()

# 14. `groupBy`: from individual rows to groups

This is the first major conceptual step.

Suppose we want:

> How many events did each user generate?

The original rows are:

```text
u1 login
u1 download
u2 login
u2 download
u3 login
u3 download
```

Mentally group them:

```text
u1 → [login, download]
u2 → [login, download]
u3 → [login, download]
```

Then count each group:

```text
u1 → 2
u2 → 2
u3 → 2
```

That is what `groupBy(...).count()` expresses.

In [ ]:
events.groupBy("user_id").count().show()

## 14.1 Aggregation

Now ask:

> How many total MB are associated with each user?

Conceptually:

```text
u1 → 10 + 50 = 60
u2 → 12 + 70 = 82
u3 →  9 + 80 = 89
```

In [ ]:
events.groupBy("user_id").agg(
    F.sum("bytes_mb").alias("total_mb"),
    F.avg("bytes_mb").alias("avg_mb"),
    F.count("*").alias("n_events"),
).show()

## 14.2 Why `groupBy` matters for SDS

This exact shape appears repeatedly:

```text
many distributed rows
      ↓
group rows by a key
      ↓
combine values for that key
```

Later examples:

- PageRank: group contributions by destination node.
- HITS authority: group hub scores by destination.
- HITS hub: group authority scores by source.
- graph degree: group edges by node.
- triangle/clustering computations: aggregate path/closure counts.

So `groupBy + agg` is not just syntax. It is one of the central computational patterns of the exam.

# 15. Part 1 checkpoint — guided first, independent second

## Guided Q1

What does this do?

```python
events.filter(F.col("country") == "PH")
```

A. sends all data to Python  
B. describes a DataFrame containing PH rows  
C. permanently deletes non-PH rows from the original DataFrame

**Answer:** B.

## Guided Q2

What does this do?

```python
events.groupBy("user_id").agg(F.sum("bytes_mb"))
```

A. one output row per event  
B. one output row per user with an aggregate  
C. turns the DataFrame into a Python dictionary

**Answer:** B.

## Guided Q3

Which operation is most dangerous on a huge DataFrame if you have not first made the result small?

A. `select`  
B. `filter`  
C. `collect`

**Answer:** C.

## Independent checkpoint

Without looking back, explain:

1. Why Spark exists.
2. Driver vs executor.
3. What a partition is.
4. Transformation vs action.
5. Why lazy evaluation is useful at a high level.
6. Why `count()` can be safe even when the underlying DataFrame is huge.
7. Why `collect()` can be unsafe.
8. What `select` changes.
9. What `filter` changes.
10. What `groupBy + agg` changes.

Then write Spark code for:

- PH events only;
- `user_id` and `bytes_mb` only;
- total MB per user.

Do not continue to Part 2 until these ideas feel ordinary.

# 16. Part 1 summary

The six sentences to remember:

1. **Spark distributes large data and computation.**
2. **The driver coordinates; executors perform distributed work.**
3. **Partitions are pieces of distributed data/work.**
4. **Transformations describe work; actions trigger work.**
5. **A DataFrame variable is not the same as a Python list containing every row.**
6. **`collect()` brings the entire result to the driver, so only collect results known to be small.**

Part 2 will build on this foundation with the most important practical exam skill: **joins**.